# TechSmart Capstone Project

## 📋 Overview 
This is the TechSmart Capstone Project. It is using SQL with Python to analyze the TechSmart company.


## 📚 Dataset Information
The <b>TechSmart</b> dataset is used, which contains information about a retail chain's operations across North America and Europe. The dataset includes:
- <b>Employee_Records:</b> Information about employees, their roles, and sales performance;
- <b>Product_Details:</b> Details about products, including categories and inventory;
- <b>Customer_Demographics:</b> Customer information and loyalty program status;
- <b>Sales_Transactions:</b> Transaction data linking customers, products, and employees.


## 🖥️ Activities

### Activity 1: Data Exploration and Cleaning 


<b>Step 1:</b> Connecting to the database, then loading and displaying tables:

In [3]:
import sqlite3
import pandas as pd

# Setting up the database
from techsmart_database import setup_database
setup_database() 
conn = sqlite3.connect('techsmart.db')

# Loading and displaying tables
tables = ['Employee_Records', 'Product_Details', 'Customer_Demographics', 'Sales_Transactions']
for table in tables:
    query = f"SELECT * FROM {table} LIMIT 10"
    df = pd.read_sql_query(query, conn)
    print(f"\n{table}:")
    display(df)

✅ Database setup complete: Tables created and populated with data!

Employee_Records:


,employee_id,role,store_location,sales_performance
0,1,Cashier,New York,None
1,2,Cashier,Los Angeles,3000
2,3,Supervisor,Phoenix,None
3,4,Manager,Chicago,3000
4,5,Manager,Phoenix,3000
5,6,Sales Associate,Paris,3000
6,7,Sales Associate,New York,2000
7,8,Cashier,London,3000
8,9,Cashier,Chicago,2000
9,10,Supervisor,Houston,1000



Product_Details:


,product_id,product_name,category,price,stock
0,101,Camera,Electronics,150,50
1,102,Charger,Accessories,40,500
2,103,Monitor,Electronics,500,100
3,104,Headphones,Electronics,200,100
4,105,Speaker,Accessories,40,300
5,106,Smartphone,Electronics,one hundred,500
6,107,Tablet,Accessories,150,500
7,108,Laptop,Electronics,one hundred,150
8,109,Mouse,Electronics,25,300
9,110,Keyboard,Electronics,fifty,300



Customer_Demographics:


,customer_id,age,gender,location,loyalty_program
0,1,35,F,San Antonio,None
1,2,50,M,London,No
2,3,37,M,Austin,No
3,4,None,M,San Antonio,None
4,5,28,M,San Antonio,None
5,6,25,M,Los Angeles,Yes
6,7,43,F,New York,Yes
7,8,None,M,San Jose,No
8,9,42,M,Paris,No
9,10,41,F,Austin,None



Sales_Transactions:


,transaction_id,customer_id,product_id,employee_id,quantity,total_amount,sale_date
0,1,13,105,2.0,4,90.0,2025-03-03 00:00:00
1,2,37,105,51.0,2,90.0,2025-03-08 00:00:00
2,3,5,106a,NaN,three,NaN,2025-03-01 00:00:00
3,4,35,109b,22.0,three,40.0,2025-03-04 00:00:00
4,5,15,101,45.0,2,NaN,2025-03-06 00:00:00
5,6,4,103,25.0,1,40.0,2025-03-09 00:00:00
6,7,37,106a,45.0,2,60.0,2025-03-06 00:00:00
7,8,13,102,39.0,three,NaN,2025-03-05 00:00:00
8,9,13,104,22.0,2,40.0,2025-03-04 00:00:00
9,10,42,101,15.0,2,60.0,2025-03-08 00:00:00


<b>Step 2:</b> Identifying and handling missing values in tables:

In [2]:
# Verifying Employee_Records
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN sales_performance IS NULL OR sales_performance = 'nan' THEN 1 ELSE 0 END) as missing_sales
FROM Employee_Records
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_sales
0,100,11


In [3]:
# Verifying Product_Details
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN stock IS NULL OR stock = 'NaN' THEN 1 ELSE 0 END) as missing_stock
FROM Product_Details
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_stock
0,10,0


In [4]:
# Verifying Customer_Demographics
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN loyalty_program IS NULL OR age IS NULL THEN 1 ELSE 0 END) as missing_demographics
FROM Customer_Demographics
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_demographics
0,100,43


In [5]:
# Verifying Sales_Transactions
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN customer_id = 'NaN' OR
       employee_id = 'NaN' OR
       quantity IS NULL OR
       total_amount = 'NaN' THEN 1 ELSE 0 END) as missing_transactions
FROM Sales_Transactions
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_transactions
0,100,17


<b>Step 3:</b> Cleaning inconsistent data formats:

In [6]:
# Cleaning Employee_Records
query = """
UPDATE Employee_Records
SET sales_performance = CASE
    WHEN sales_performance = 'nan' THEN NULL
    WHEN sales_performance = 'five thousand' THEN '5000'
    ELSE sales_performance
END
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [7]:
# Cleaning Product_Details
query = """
UPDATE Product_Details
SET price = CASE
      WHEN price = 'nan' THEN NULL
      WHEN price = 'fifty' THEN '50'
      WHEN price = 'one hundred' THEN '100'
      ELSE price
END,
    product_id = SUBSTR( product_id, 1, 3 )
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [8]:
# Cleaning Customer_Demographics
query = """
UPDATE Customer_Demographics
SET age = CASE
      WHEN age IS NULL THEN NULL
      WHEN age = 'thirty-five' THEN '35'
      ELSE age
END
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [9]:
# Cleaning Sales_Transactions
query = """
UPDATE Sales_Transactions
SET
    product_id = SUBSTR(product_id, 1, 3),
    customer_id = NULLIF(customer_id, 'NaN'),
    employee_id = NULLIF(employee_id, 'NaN'),
    quantity = CASE
        WHEN quantity = 'three' THEN '3'
        WHEN quantity = 'NaN' THEN NULL
        ELSE quantity
    END,
    total_amount = NULLIF(total_amount, 'NaN');
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

### Activity 2: Advanced Data Analysis  


<b>Step 1:</b> Analyzing employee performance by location:

In [10]:
query = """
WITH emp_sales AS (
    SELECT store_location, 
           AVG(CAST(sales_performance AS FLOAT)) as avg_sales,
           COUNT(*) as employee_count
    FROM Employee_Records
    WHERE sales_performance IS NOT NULL
    GROUP BY store_location
)
SELECT store_location, avg_sales, employee_count,
       RANK() OVER (ORDER BY avg_sales DESC) as location_rank
FROM emp_sales
ORDER BY avg_sales DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,store_location,avg_sales,employee_count,location_rank
0,Chicago,3875.000000,8,1
1,Phoenix,3312.500000,16,2
2,London,3214.285714,14,3
3,New York,2823.529412,17,4
4,Los Angeles,2692.307692,13,5
5,Paris,2600.000000,10,6
6,Houston,2363.636364,11,7


<b>Step 2:</b> Identifying top-selling products by category:

In [11]:
query = """
SELECT *
FROM (
    SELECT p.category,
           p.product_name,
           SUM(s.quantity) AS total_sold,
           RANK() OVER (
               PARTITION BY p.category
               ORDER BY SUM(s.quantity) DESC
           ) AS rank_in_category
    FROM Sales_Transactions s
    JOIN Product_Details p ON s.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
WHERE rank_in_category <= 5
ORDER BY category, total_sold DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,category,product_name,total_sold,rank_in_category
0,Accessories,Speaker,30,1
1,Accessories,Charger,22,2
2,Electronics,Monitor,37,1
3,Electronics,Camera,36,2
4,Electronics,Headphones,33,3
5,Electronics,Smartphone,31,4
6,Electronics,Mouse,18,5


 <b>Step 3: </b>  Analyzing customer purchasing behavior:

In [5]:
query = """
SELECT *
FROM (
    SELECT d.gender,
           d.age,
           SUM(s.quantity) AS total_sold,
           RANK() OVER (
               PARTITION BY d.gender
               ORDER BY SUM(s.quantity) DESC
           ) AS rank_in_gender
    FROM Sales_Transactions s
    JOIN Customer_Demographics d ON s.customer_id = d.customer_id
    GROUP BY d.gender, d.age
)
ORDER BY gender, total_sold DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,gender,age,total_sold,rank_in_gender
0,F,37,22.0,1
1,F,55,19.0,2
2,F,30,15.0,3
3,F,20,12.0,4
4,F,40,10.0,5
5,F,43,4.0,6
6,F,41,4.0,6
7,F,35,3.0,8
8,M,37,25.0,1
9,M,22,10.0,2


### Activity 3: Performance Optimizition 

As a dataset grows, query performance becomes crucial. The below query analyzes sales performance for Electronics and Accessories by summarizing transactions per employee, store, and customer loyalty status, while also computing total revenue per store-category pair for ranking and comparison.

<b>Step 1:</b> Measuring time of a complex query:

In [13]:
# Measuring time of a complex query
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;
"""

for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.5073375701904297 seconds


In [14]:
# Displaying the results of the query executed 100 times
display(df)

,store_location,employee_id,role,category,loyalty_program,total_sales,total_units_sold,total_revenue,store_revenue
0,Chicago,17,Sales Associate,Electronics,None,4,8.0,160.0,200.0
1,Chicago,17,Sales Associate,Accessories,No,2,3.0,90.0,130.0
2,Chicago,17,Sales Associate,Accessories,None,1,1.0,40.0,130.0
3,Chicago,17,Sales Associate,Electronics,No,1,1.0,40.0,200.0
4,Houston,45,Manager,Accessories,No,3,6.0,220.0,310.0
5,Houston,45,Manager,Electronics,None,3,7.0,210.0,330.0
6,Houston,45,Manager,Electronics,Yes,4,8.0,120.0,330.0
7,Houston,45,Manager,Accessories,None,1,NaN,90.0,310.0
8,London,37,Cashier,Accessories,None,2,4.0,140.0,140.0
9,London,37,Cashier,Electronics,None,3,2.0,120.0,190.0


<b>Step 2:</b> Creating indexes to improve performance (4 different indexes):

In [15]:
# Creating the idx_store_location index (A)
cursor.execute("CREATE INDEX idx_store_location ON Employee_Records (store_location)")
conn.commit()

In [16]:
# Timing for the complex query executed 100 times
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""

for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.518864393234253 seconds


In [17]:
# Dropping the index
cursor.execute("DROP INDEX idx_store_location")
conn.commit()

In [18]:
# Creating the idx_employee_id_location index (B)
cursor.execute("CREATE INDEX idx_employee_id_location ON Employee_Records (employee_id, store_location)")
conn.commit()

In [19]:
# Timing for the complex query executed 100 times
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""

for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.4863319396972656 seconds


In [20]:
# Dropping the index
cursor.execute("DROP INDEX idx_employee_id_location")
conn.commit()

In [26]:
# Creating the idx_employee_id_location index (C)
cursor.execute("CREATE INDEX idx_sales_prod_emp_cust ON Sales_Transactions (product_id, employee_id, customer_id)")
conn.commit()

In [27]:
# Timing for the complex query executed 100 times
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""

for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.458343267440796 seconds


In [28]:
# Dropping the index
cursor.execute("DROP INDEX idx_sales_prod_emp_cust")
conn.commit()

In [29]:
# Creating the idx_customer_id_loyalty index (D)
cursor.execute("CREATE INDEX idx_customer_id_loyalty ON Customer_Demographics (customer_id, loyalty_program )")
conn.commit()

In [30]:
# Timing for the complex query executed 100 times
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""


for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.5215330123901367 seconds


In [31]:
# Dropping the index
cursor.execute("DROP INDEX idx_customer_id_loyalty")
conn.commit()

<b>Step 3:</b> Using an AI to further optimize the query:

In [33]:
# Using optimized by AI (Gemini) query and comparing execution time

start_time = time.time()
query = """
SELECT
    e.store_location,
    e.employee_id,
    e.role,
    p.category,
    c.loyalty_program,
    COUNT(s.transaction_id) AS total_sales,
    SUM(s.quantity) AS total_units_sold,
    SUM(s.total_amount) AS total_revenue,
    SUM(SUM(s.total_amount)) OVER (
        PARTITION BY e.store_location, p.category
    ) AS store_revenue
FROM Sales_Transactions s
JOIN Employee_Records e 
    ON s.employee_id = e.employee_id
JOIN Product_Details p 
    ON s.product_id = p.product_id
JOIN Customer_Demographics c 
    ON s.customer_id = c.customer_id
WHERE p.category IN ('Electronics', 'Accessories')
GROUP BY 
    e.store_location, 
    e.employee_id, 
    e.role, 
    p.category, 
    c.loyalty_program
ORDER BY 
    e.store_location, 
    total_revenue DESC;
"""

for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.5716068744659424 seconds


<b>Step 4:</b> Deleting the rows with NULL fields to improve performance:

In [35]:
# Deleting rows with NULL fields: Employee_Records
query = """
DELETE FROM Employee_Records
WHERE (sales_performance IS NULL);
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [36]:
# Deleting rows with NULL fields: Product_details
query = """
DELETE FROM Product_Details
WHERE (price IS NULL) OR (stock Is NULL);
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [37]:
# Deleting rows with NULL fields: Customer_Demographics
query = """
DELETE FROM Customer_Demographics
WHERE (age IS NULL) OR (loyalty_program IS NULL);
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [38]:
# Deleting rows with NULL fields: Sales_Transactions
query = """
DELETE FROM Sales_Transactions
WHERE (customer_id IS NULL) OR (employee_id IS NULL) OR (quantity IS NULL) OR (total_amount Is NULL);
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [39]:
# Displaying tables after deleting rows
tables = ['Employee_Records', 'Product_Details', 'Customer_Demographics','Sales_Transactions']
for table in tables:
    query = f"SELECT * FROM {table} LIMIT 10"
    df = pd.read_sql_query(query, conn)
    print(f"\n{table}:")
    display(df)


Employee_Records:


,employee_id,role,store_location,sales_performance
0,2,Cashier,Los Angeles,3000
1,4,Manager,Chicago,3000
2,5,Manager,Phoenix,3000
3,6,Sales Associate,Paris,3000
4,7,Sales Associate,New York,2000
5,8,Cashier,London,3000
6,9,Cashier,Chicago,2000
7,10,Supervisor,Houston,1000
8,11,Sales Associate,Paris,2000
9,12,Sales Associate,Paris,3000



Product_Details:


,product_id,product_name,category,price,stock
0,101,Camera,Electronics,150,50
1,102,Charger,Accessories,40,500
2,103,Monitor,Electronics,500,100
3,104,Headphones,Electronics,200,100
4,105,Speaker,Accessories,40,300
5,106,Smartphone,Electronics,100,500
6,107,Tablet,Accessories,150,500
7,108,Laptop,Electronics,100,150
8,109,Mouse,Electronics,25,300
9,110,Keyboard,Electronics,50,300



Customer_Demographics:


,customer_id,age,gender,location,loyalty_program
0,2,50,M,London,No
1,3,37,M,Austin,No
2,6,25,M,Los Angeles,Yes
3,7,43,F,New York,Yes
4,9,42,M,Paris,No
5,11,40,M,Austin,No
6,12,37,M,San Antonio,Yes
7,15,30,F,Houston,Yes
8,18,35,M,Chicago,No
9,19,20,M,San Antonio,Yes



Sales_Transactions:


,transaction_id,customer_id,product_id,employee_id,quantity,total_amount,sale_date
0,1,13,105,2,4,90.0,2025-03-03 00:00:00
1,2,37,105,51,2,90.0,2025-03-08 00:00:00
2,4,35,109,22,3,40.0,2025-03-04 00:00:00
3,6,4,103,25,1,40.0,2025-03-09 00:00:00
4,7,37,106,45,2,60.0,2025-03-06 00:00:00
5,9,13,104,22,2,40.0,2025-03-04 00:00:00
6,10,42,101,15,2,60.0,2025-03-08 00:00:00
7,11,41,109,3,4,40.0,2025-03-09 00:00:00
8,12,42,101,22,1,90.0,2025-03-05 00:00:00
9,14,12,104,15,4,50.0,2025-03-02 00:00:00


In [40]:
# Measuring time of a complex query after deleting the rows with NULL fields executed 100 times
import time

start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue,
   sr.store_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""
for i in range(100):
    df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 2.4903275966644287 seconds


In [41]:
# Displaying the results of the query 
display(df)

,store_location,employee_id,role,category,loyalty_program,total_sales,total_units_sold,total_revenue,store_revenue
0,Chicago,17,Sales Associate,Accessories,No,1,3,50.0,50.0
1,Chicago,17,Sales Associate,Electronics,No,1,1,40.0,40.0
2,Los Angeles,15,Manager,Electronics,Yes,3,9,200.0,720.0
3,Los Angeles,22,Sales Associate,Electronics,Yes,3,6,190.0,720.0
4,Los Angeles,25,Manager,Electronics,Yes,2,4,130.0,720.0
5,Los Angeles,15,Manager,Electronics,No,2,8,110.0,720.0
6,Los Angeles,25,Manager,Electronics,No,1,4,50.0,720.0
7,Los Angeles,2,Cashier,Electronics,No,1,2,40.0,720.0
8,Los Angeles,22,Sales Associate,Accessories,No,1,3,30.0,30.0
9,New York,39,Manager,Accessories,No,1,1,90.0,90.0


#### Close the Connection and Drop the Database


In [42]:
# Closing the database connection and drop the database file
conn.close()

from pathlib import Path
db_path = Path("techsmart.db")
db_path.unlink(missing_ok=True)